# Основная модель: CatBoost (Pool + тюнинг)
Цель — получить сильную базовую модель и аккуратно подобрать гиперпараметры.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)
DATA_PATH = (
    PROJECT_ROOT
    / "Data_making"
    / "synthetic_education_dushanbe_WITH_ROUNDED.csv"
)
METRICS_DIR = PROJECT_ROOT / "Models" / "Compare models"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
from catboost import CatBoostClassifier, Pool, cv
from sklearn.model_selection import train_test_split, ParameterGrid

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Общий набор признаков для честного сравнения моделей
TARGET_COL = "Рез_экзамена"
ID_COL = "ID_ученика"
LEAKAGE_RISK_COLS = ["Средний_балл"]
FEATURE_COLUMNS = [
    "Класс",
    "Район",
    "Часы_самоподготовки_в_неделю",
    "Посещаемость_%",
    "Уверенность_в_себе",
    "Уровень_стресса_перед_экзаменом",
    "Пропущенные_дни",
    "Тип_школы",
    "Индекс_качества_школы",
    "Стабильность_преподавателей",
    "Доступ_к_ресурсам",
    "Образовательная_среда",
]
y = df[TARGET_COL]
X = df[FEATURE_COLUMNS].copy()

# Проверка бинарности и баланса
unique_vals = np.sort(y.unique())
print('Target unique:', unique_vals)
print('Class balance:\n', y.value_counts(normalize=True))

if len(unique_vals) != 2:
    raise ValueError('Ожидается бинарный таргет для CatBoostClassifier.')

pos_weight = (y == 0).sum() / (y == 1).sum()
class_weights = [1.0, float(pos_weight)]

In [ ]:
# Категориальные признаки
categorical_features = X.select_dtypes(include=['object']).columns
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_features]

# Train/Valid split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_indices)

## Базовый CatBoost
Стартовая модель для ориентира перед тюнингом.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, classification_report
)

In [ ]:
base_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    iterations=500,
    class_weights=class_weights,
    verbose=False
)

base_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, plot=True)
base_pred = base_model.predict_proba(X_valid)[:, 1]

print('Base AUC:', roc_auc_score(y_valid, base_pred))



# Предсказания вероятностей и классов
base_pred_proba = base_model.predict_proba(X_valid)[:, 1]
base_pred_class = (base_pred_proba >= 0.5).astype(int)

# Метрики
print("Base Accuracy:", accuracy_score(y_valid, base_pred_class))
print("Base Precision:", precision_score(y_valid, base_pred_class))
print("Base Recall:", recall_score(y_valid, base_pred_class))
print("Base F1:", f1_score(y_valid, base_pred_class))
print("Base ROC-AUC:", roc_auc_score(y_valid, base_pred_proba))
print("Base LogLoss:", log_loss(y_valid, base_pred_proba))

# Дополнительно: подробный отчёт по классам
print("\nClassification Report:\n", classification_report(y_valid, base_pred_class))

In [ ]:
# Важность признаков базовой модели (для анализа)
base_importance = base_model.get_feature_importance(type='FeatureImportance')
base_fi_df = (
    pd.DataFrame({'feature': X.columns, 'importance': base_importance})
      .sort_values('importance', ascending=False)
      .reset_index(drop=True)
)

# Optional extended feature experiment (не используется в основном benchmark):
# df_fe['Бал_на_стресс'] = ...
# df_fe['Бал_на_уверенность'] = ...
# df_fe['Балл_минус_стресс'] = ...
# df_fe['Пропуски_на_стресс'] = ...
# df_fe['Среда_школы_среднее'] = ...
# df_fe['Качество_на_среду'] = ...
# df_fe['Качество_на_ресурсы'] = ...

# Основной benchmark использует только общий набор из 12 исходных признаков.
y = df[TARGET_COL]
X = df[FEATURE_COLUMNS].copy()

# Переобновляем категориальные признаки и сплит
categorical_features = X.select_dtypes(include=['object']).columns
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_features]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_indices)


## Тюнинг гиперпараметров
Используем `catboost.cv` (5‑fold) с early stopping. Это медленнее, но надежнее.


In [ ]:
base_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': False,
    'od_type': 'Iter',
    'od_wait': 50,
    'class_weights': class_weights
}

param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [400, 800, 1200],
    'bagging_temperature': [0, 1, 3],
    'random_strength': [1, 2, 5]
}

In [ ]:
# Сколько комбинаций реально прогнать
max_evals = 20
grid = list(ParameterGrid(param_grid))
np.random.shuffle(grid)
grid = grid[:max_evals]

best_auc = -1
best_params = None

for params in grid:
    cv_params = {**base_params, **params}
    cv_data = cv(
        pool=train_pool,
        params=cv_params,
        fold_count=5,
        stratified=True,
        early_stopping_rounds=50,
        verbose=False
    )
    auc = cv_data['test-AUC-mean'].max()

    if auc > best_auc:
        best_auc = auc
        best_params = params

print('Best CV AUC:', best_auc)
print('Best params:', best_params)

## Финальная модель и метрики
Обучаем финальную модель на лучших параметрах и оцениваем на valid.


In [ ]:
if best_params is None:
    best_params = {}

final_model = CatBoostClassifier(
    **base_params,
    **best_params
)

final_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
final_pred = final_model.predict_proba(X_valid)[:, 1]

# Подбор порога: максимизируем Recall при ограничении на Precision
TARGET_MIN_PRECISION = 0.80
threshold_grid = np.linspace(0.1, 0.9, 161)

best_threshold = 0.5
best_recall = -1
best_precision = -1

for thr in threshold_grid:
    pred_thr = (final_pred >= thr).astype(int)
    p = precision_score(y_valid, pred_thr, zero_division=0)
    r = recall_score(y_valid, pred_thr, zero_division=0)

    if p >= TARGET_MIN_PRECISION and r > best_recall:
        best_recall = r
        best_precision = p
        best_threshold = float(thr)

# fallback: если ограничение слишком жесткое, берем порог с максимальным Recall
if best_recall < 0:
    for thr in threshold_grid:
        pred_thr = (final_pred >= thr).astype(int)
        p = precision_score(y_valid, pred_thr, zero_division=0)
        r = recall_score(y_valid, pred_thr, zero_division=0)
        if r > best_recall:
            best_recall = r
            best_precision = p
            best_threshold = float(thr)

final_threshold = best_threshold
final_pred_label = (final_pred >= final_threshold).astype(int)

print('Final AUC:', roc_auc_score(y_valid, final_pred))
print('Selected threshold:', round(final_threshold, 3))
print('Accuracy:', accuracy_score(y_valid, final_pred_label))
print('Precision:', precision_score(y_valid, final_pred_label, zero_division=0))
print('Recall:', recall_score(y_valid, final_pred_label, zero_division=0))
print('F1:', f1_score(y_valid, final_pred_label, zero_division=0))


## Визуализация результатов (CatBoost)
Графики ключевых метрик, ROC и матрицы ошибок для понятной интерпретации.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, confusion_matrix

In [ ]:
# Гистограмма вероятностей (насколько модель уверена)
fig_p = px.histogram(x=final_pred, nbins=30, title='Распределение предсказанных вероятностей')
fig_p.update_layout(xaxis_title='P(класс=1)', yaxis_title='Количество')
fig_p.show()

In [ ]:
# Основные метрики
metrics = {
    'AUC': roc_auc_score(y_valid, final_pred),
    'Accuracy': accuracy_score(y_valid, final_pred_label),
    'Precision': precision_score(y_valid, final_pred_label),
    'Recall': recall_score(y_valid, final_pred_label),
    'F1': f1_score(y_valid, final_pred_label)
}
metrics_df = pd.DataFrame({'metric': list(metrics.keys()), 'value': list(metrics.values())})
fig_m = px.bar(metrics_df, x='metric', y='value', text='value', title='Ключевые метрики CatBoost')
fig_m.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_m.update_layout(yaxis=dict(range=[0, 1]))
fig_m.show()

In [ ]:
# ROC-кривая
fpr, tpr, _ = roc_curve(y_valid, final_pred)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC'))
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig_roc.update_layout(title='ROC-кривая (CatBoost)', xaxis_title='FPR', yaxis_title='TPR')
fig_roc.show()

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_valid, final_pred_label)
fig_cm = px.imshow(cm, text_auto=True, aspect='auto',
                   labels=dict(x='Predicted', y='Actual', color='Count'),
                   title='Матрица ошибок (CatBoost)')
fig_cm.show()

## SHAP-анализ (интерактивный Plotly)
Ниже интерактивные визуализации вклада признаков для финальной модели CatBoost на валидационной выборке.


In [ ]:
# SHAP values для CatBoost + интерактивные графики Plotly
from plotly.subplots import make_subplots

# 1) SHAP матрица: (n_samples, n_features + 1), последний столбец — expected value
shap_raw = final_model.get_feature_importance(valid_pool, type='ShapValues')
shap_values = shap_raw[:, :-1]
expected_value = shap_raw[:, -1]

feature_names = X_valid.columns.tolist()
shap_df = pd.DataFrame(shap_values, columns=feature_names, index=X_valid.index)

# 2) Глобальная важность (mean |SHAP|)
global_shap = shap_df.abs().mean().sort_values(ascending=False)
top_n = min(12, len(global_shap))
top_features = global_shap.head(top_n).index.tolist()

fig_shap_bar = px.bar(
    global_shap.head(top_n).sort_values(),
    orientation='h',
    labels={'value': 'mean(|SHAP|)', 'index': 'Признак'},
    title='SHAP: глобальная важность признаков (Top-12)',
    color=global_shap.head(top_n).sort_values().values,
    color_continuous_scale='Tealgrn'
)
fig_shap_bar.update_layout(template='plotly_white', coloraxis_showscale=False, height=520)
fig_shap_bar.show()

# 3) Beeswarm-style интерактивный scatter (Top features)
rng = np.random.default_rng(42)
sample_size = min(1500, len(X_valid))
sample_pos = rng.choice(np.arange(len(X_valid)), size=sample_size, replace=False)
Xv = X_valid.reset_index(drop=True).iloc[sample_pos].copy()
Sv = shap_df.reset_index(drop=True).iloc[sample_pos].copy()

rows = []
for rank, feat in enumerate(top_features, start=1):
    feat_vals = Xv[feat]
    shap_vals = Sv[feat].values

    if pd.api.types.is_numeric_dtype(feat_vals):
        v = feat_vals.astype(float).values
        vmin, vmax = np.nanmin(v), np.nanmax(v)
        color_norm = (v - vmin) / (vmax - vmin + 1e-9)
        feat_display = feat_vals.round(4).astype(str).values
    else:
        cat_codes = pd.Categorical(feat_vals).codes.astype(float)
        cmax = max(cat_codes.max(), 1.0)
        color_norm = cat_codes / cmax
        feat_display = feat_vals.astype(str).values

    y_base = top_n - rank + 1
    y_jitter = y_base + rng.normal(0, 0.16, size=len(shap_vals))

    rows.append(pd.DataFrame({
        'feature': feat,
        'feature_value': feat_display,
        'shap_value': shap_vals,
        'y': y_jitter,
        'color_norm': color_norm,
    }))

beeswarm_df = pd.concat(rows, ignore_index=True)

fig_bee = go.Figure(
    go.Scattergl(
        x=beeswarm_df['shap_value'],
        y=beeswarm_df['y'],
        mode='markers',
        marker=dict(
            size=6,
            color=beeswarm_df['color_norm'],
            colorscale='RdBu_r',
            cmin=0,
            cmax=1,
            opacity=0.72,
            colorbar=dict(title='Значение фичи (норм.)')
        ),
        customdata=np.stack([beeswarm_df['feature'], beeswarm_df['feature_value']], axis=1),
        hovertemplate='Признак: %{customdata[0]}<br>SHAP: %{x:.4f}<br>Значение: %{customdata[1]}<extra></extra>'
    )
)

fig_bee.update_layout(
    title='SHAP Beeswarm (интерактивный, Top-12)',
    template='plotly_white',
    height=640,
    xaxis_title='SHAP value (вклад в вероятность класса 1)',
    yaxis=dict(
        title='Признаки',
        tickmode='array',
        tickvals=list(range(1, top_n + 1)),
        ticktext=top_features[::-1]
    )
)
fig_bee.show()

# 4) Dependence plot с dropdown (числовые фичи)
numeric_top = [f for f in top_features if pd.api.types.is_numeric_dtype(X_valid[f])][:8]
if numeric_top:
    fig_dep = go.Figure()

    for i, feat in enumerate(numeric_top):
        fig_dep.add_trace(
            go.Scattergl(
                x=X_valid[feat],
                y=shap_df[feat],
                mode='markers',
                marker=dict(
                    size=7,
                    color=final_pred,
                    colorscale='Viridis',
                    opacity=0.7,
                    colorbar=dict(title='P(class=1)') if i == 0 else None,
                    showscale=True if i == 0 else False
                ),
                name=feat,
                visible=(i == 0),
                hovertemplate=f'Признак: {feat}<br>Значение: %{{x}}<br>SHAP: %{{y:.4f}}<br>P(1): %{{marker.color:.3f}}<extra></extra>'
            )
        )

    buttons = []
    for i, feat in enumerate(numeric_top):
        vis = [False] * len(numeric_top)
        vis[i] = True
        buttons.append(dict(
            label=feat,
            method='update',
            args=[
                {'visible': vis},
                {'title': f'SHAP Dependence: {feat}', 'xaxis': {'title': feat}}
            ]
        ))

    fig_dep.update_layout(
        template='plotly_white',
        height=560,
        title=f'SHAP Dependence: {numeric_top[0]}',
        xaxis_title=numeric_top[0],
        yaxis_title='SHAP value',
        updatemenus=[dict(buttons=buttons, direction='down', x=1.02, y=1.0, xanchor='left', yanchor='top')]
    )
    fig_dep.show()
else:
    print('Для dependence plot не найдено числовых фич в Top-важных признаках.')

# 5) Таблица Top SHAP для отчета
display(global_shap.head(20).to_frame('mean_abs_shap').round(6))
print('SHAP expected value (mean):', float(np.mean(expected_value)))

## Сохранение метрик для сравнения


In [ ]:
import json
cat_metrics = {
    'model': 'CatBoost',
    'AUC': float(roc_auc_score(y_valid, final_pred)),
    'Accuracy': float(accuracy_score(y_valid, final_pred_label)),
    'Precision': float(precision_score(y_valid, final_pred_label, zero_division=0)),
    'Recall': float(recall_score(y_valid, final_pred_label, zero_division=0)),
    'F1': float(f1_score(y_valid, final_pred_label, zero_division=0)),
    'threshold': float(final_threshold),
    'threshold_strategy': 'maximize_recall_with_min_precision',
    'min_precision_constraint': 0.80,
    'feature_scenario': 'common_no_avg_grade',
    'feature_columns': FEATURE_COLUMNS,
    'excluded_leakage_risk_cols': LEAKAGE_RISK_COLS
}

out_path = METRICS_DIR / "catboost_metrics.json"
out_path.write_text(json.dumps(cat_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print('saved', out_path)
